# EDA — E-commerce Fraud Data

Exploratory analysis of `Fraud_Data.csv`: cleaning overview, univariate/bivariate plots, class imbalance, and fraud patterns by country after IP geolocation merge.

**Place raw files in** `data/raw/` before running.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data_loader import load_fraud_data, load_ip_country
from src.preprocessing import (
    class_distribution,
    clean_fraud_data,
)

sns.set_theme(style="whitegrid", context="notebook")
RAW = ROOT / "data" / "raw"
print("Project root:", ROOT)

## 1. Load & clean

In [ ]:
fraud_raw = load_fraud_data(raw_dir=RAW)
ip_map = load_ip_country(raw_dir=RAW)

print("Raw shape:", fraud_raw.shape)
print("Missing values:\n", fraud_raw.isna().sum())
print("Duplicate rows:", fraud_raw.duplicated().sum())
fraud_raw.head()

In [ ]:
# Justification: drop rows with NA after dtype coercion; duplicates removed.
# IP→country merge uses vectorized range lookup (searchsorted).
fraud = clean_fraud_data(fraud_raw, ip_country_df=ip_map, drop_na=True)
print("Cleaned shape:", fraud.shape)
fraud.dtypes

## 2. Class imbalance

In [ ]:
dist = class_distribution(fraud["class"], label="fraud_class")
display(dist)

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=fraud, x="class", ax=ax, palette="Set2")
ax.set_title("Class imbalance (0=legit, 1=fraud)")
plt.show()

fraud_rate = fraud["class"].mean()
print(f"Fraud rate: {fraud_rate:.2%} — accuracy alone is misleading.")

## 3. Univariate distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(fraud["purchase_value"], bins=40, ax=axes[0, 0], kde=True)
axes[0, 0].set_title("purchase_value")
sns.histplot(fraud["age"], bins=30, ax=axes[0, 1], kde=True)
axes[0, 1].set_title("age")
sns.countplot(data=fraud, x="source", ax=axes[1, 0], order=fraud["source"].value_counts().index)
axes[1, 0].set_title("source")
sns.countplot(data=fraud, x="browser", ax=axes[1, 1], order=fraud["browser"].value_counts().index)
axes[1, 1].set_title("browser")
plt.tight_layout()
plt.show()

## 4. Bivariate — features vs fraud

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.boxplot(data=fraud, x="class", y="purchase_value", ax=axes[0])
axes[0].set_title("purchase_value by class")
sns.boxplot(data=fraud, x="class", y="age", ax=axes[1])
axes[1].set_title("age by class")
pd.crosstab(fraud["sex"], fraud["class"], normalize="index").plot(
    kind="bar", ax=axes[2], title="Fraud rate by sex"
)
plt.tight_layout()
plt.show()

print("Fraud rate by source:")
display(fraud.groupby("source")["class"].mean().sort_values(ascending=False))

## 5. Geolocation — fraud by country

In [ ]:
by_country = (
    fraud.groupby("country")
    .agg(n=("class", "size"), fraud_rate=("class", "mean"), fraud_n=("class", "sum"))
    .query("n >= 50")
    .sort_values("fraud_rate", ascending=False)
)
display(by_country.head(15))

fig, ax = plt.subplots(figsize=(10, 5))
top = by_country.head(12)
sns.barplot(x=top.index, y=top["fraud_rate"], ax=ax, palette="Reds_r")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
ax.set_title("Fraud rate by country (n ≥ 50)")
plt.tight_layout()
plt.show()

## EDA takeaways

- Severe class imbalance → prefer AUC-PR / F1 over accuracy.
- Country and channel (`source`) show heterogeneous fraud rates — useful features.
- Next: feature engineering (`feature-engineering.ipynb`).